# RTE teacher / baseline: DeBERTa-v3-base on Colab

Runs the **same code and pinned packages** as the local venv. Produces:
1. `outputs/baseline_deberta_v3_base/`: baseline metrics (5 seeds), dev predictions, best-val checkpoint
2. `outputs/baseline_deberta_v3_base_oof/teacher_logits_train.npy`: out-of-fold soft labels for distillation

Runtime → Change runtime type → **GPU** (T4 is enough; L4/A100 is faster).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import sys; print(sys.version)

## 1. Get the code
Either clone your repo (set `REPO_URL`) or upload `rte-challenge.zip` (the project without `.venv/`, `data/`, `outputs/`).

In [ ]:
REPO_URL = ""  # e.g. https://github.com/<you>/rte-challenge.git ; leave empty to upload a zip

%cd /content
if REPO_URL:
    !git clone -q $REPO_URL rte-challenge
else:
    from google.colab import files
    up = files.upload()  # choose rte-challenge.zip
    !unzip -q -o {next(iter(up))} -d /content
%cd /content/rte-challenge

## 2. Install pinned dependencies (keep Colab's CUDA torch)
The universal lock is filtered so Colab's preinstalled CUDA build of torch (and its nvidia/triton libs) is untouched. If pip reports that numpy or another core package changed, use **Runtime → Restart session** and continue from the next cell.

In [ ]:
!grep -vE '^(torch|nvidia-|triton|cuda-)' requirements.txt > /tmp/req-colab.txt
!pip install -q -r /tmp/req-colab.txt
!pip install -q --no-deps -e .
import torch, transformers; print(torch.__version__, transformers.__version__, torch.cuda.is_available())

## 3. Data
Upload the challenge zip (`Dataset for Challenge-20261101.zip`). The dev split is fetched from GLUE RTE validation.

In [ ]:
%cd /content/rte-challenge
from google.colab import files
up = files.upload()  # choose the challenge dataset zip
ZIP = next(iter(up))
!python scripts/prepare_data.py --zip "$ZIP" --out data
!python -m pytest -q

## 4. Baseline: DeBERTa-v3-base, 5 seeds
Each seed early-stops on a premise-grouped 10% hold-out of train; dev is scored once per seed.

In [ ]:
!python -m rte.train --config configs/baseline_deberta_v3_base.yaml

## 5. Teacher logits (out-of-fold, 5 folds grouped by premise)

In [ ]:
!python -m rte.train --config configs/baseline_deberta_v3_base.yaml --oof

## 6. Download artifacts
Unzip into the local project root; this restores `outputs/baseline_deberta_v3_base*/`. The checkpoint (~0.7 GB) is optional: drop `INCLUDE_MODEL` if you only need metrics and teacher logits.

In [ ]:
INCLUDE_MODEL = False
excl = "" if INCLUDE_MODEL else "-x 'outputs/baseline_deberta_v3_base/model/*'"
!zip -q -r teacher_artifacts.zip outputs/baseline_deberta_v3_base outputs/baseline_deberta_v3_base_oof {excl}
!du -h teacher_artifacts.zip
from google.colab import files
files.download("teacher_artifacts.zip")